### Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/05/25s4fzd90_906hkl3kjr7vxc0000gn/T/ipykernel_96169/2865240289.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader


In [2]:
# processing all the pdfs
def process_all_pdf(pdf_directory):
    """Process all the PDFs in the given directory"""
    all_docs = [] #initially empty
    pdf_dir = Path(pdf_directory)

    """ Find all the pdf files in the dir"""
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdfs in {pdf_dir} directory")

    """ Processing each file"""

    for pdf in pdf_files:
        print(f"Processing {pdf}")
        try :
            # Loading a file
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()
            # adding metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf.name
                doc.metadata["file_type"] = "pdf"

            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"error {e}")

    print(f"Total number of documents loaded: {len(all_docs)}")
    return all_docs
    


all_pdf_documents = process_all_pdf("/Users/Armaan/Desktop/learn/RAG/data/pdfs")

Found 2 pdfs in /Users/Armaan/Desktop/learn/RAG/data/pdfs directory
Processing /Users/Armaan/Desktop/learn/RAG/data/pdfs/RAG_Experience.pdf
Loaded 36 pages
Processing /Users/Armaan/Desktop/learn/RAG/data/pdfs/RAG_Survey.pdf
Loaded 21 pages
Total number of documents loaded: 57


### Text Splitting into Chunks

In [3]:
def split_documents(documents, chunk_size=1000, chunks_overlap=200):
    """Split docs into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap=chunks_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print("Example Chunk:")
        print("Content:")
        print(f"{split_docs[0].page_content[:200]}...")
        print("Metadata:")
        print(f"{split_docs[0].metadata}")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
chunks

Split 57 into 242 chunks
Example Chunk:
Content:
Developing Retrieval Augmented Generation
(RAG) based LLM Systems from PDFs: An
Experience Report
Ayman Asad Khan
Tampere University
ayman.khan@tuni.fi
Md Toufique Hasan
Tampere University
mdtoufique....
Metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-10-22T01:56:19+00:00', 'author': '', 'keywords': '', 'moddate': '2024-10-22T01:56:19+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/Armaan/Desktop/learn/RAG/data/pdfs/RAG_Experience.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'RAG_Experience.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-10-22T01:56:19+00:00', 'author': '', 'keywords': '', 'moddate': '2024-10-22T01:56:19+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/Armaan/Desktop/learn/RAG/data/pdfs/RAG_Experience.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'RAG_Experience.pdf', 'file_type': 'pdf'}, page_content='Developing Retrieval Augmented Generation\n(RAG) based LLM Systems from PDFs: An\nExperience Report\nAyman Asad Khan\nTampere University\nayman.khan@tuni.fi\nMd Toufique Hasan\nTampere University\nmdtoufique.hasan@tuni.fi\nKai Kristian Kemell\nTampere University\nkai-kristian.kemell@tuni.fi\nJussi Rasku\nTampere University\njussi.rasku@tuni.fi\nPekka Abrahamsson\nTampere University\npekka.abrahamsson@tuni.fi\nAbstract. This paper presents an ex

### Embedding and VectorstoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
import os

In [6]:
class EmbeddingManager:
    """It will handle  the generation of document embeddings via sentence transformers"""

    def __init__(self, model_name:str= "all-MiniLM-L6-v2"):
        """Initialize the embedding manager"""

        """
        Args : Huggingface model name for sentence transformers
        """
        self.model_name=model_name
        self.model = None 
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print("Trying to load the model")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str])-> np.ndarray:
        """
        Generate embeddings for a list of the text via the model
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)}  texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generate embeddins with shape of {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


Trying to load the model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding Dimension: 384


### VectorStore

In [7]:
class VectorStore:
    """Manages the vector embeddings in the vector database; chromedb in our case"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory=persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # create persistent chromDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"description" :"PDF document containing embeddings for RAG"}
                )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection :{self.collection.count()} " )
        except Exception as e:
            printf("Error while initializing the store: {e}")


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids:List[str] = []
        metadatas:List[Any] = []
        documents_text: List[Any] = []
        embeddings_list: List[List[Any]] = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise



vector_store = VectorStore()
vector_store


Vector store initialized. Collection: pdf_documents
Existing documents in collection :0 


In [8]:
texts = [ doc.page_content for doc in chunks]
texts
# storring them in chroma db
embeddings = embedding_manager.generate_embeddings(texts=texts)

vector_store.add_documents(documents=chunks,embeddings=embeddings)

Generating embeddings for 242  texts...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Generate embeddins with shape of (242, 384)
Adding 242 documents to vector store...
Successfully added 242 documents to vector store
Total documents in collection: 242


### Retrievel Pipeline from VectorStore

In [9]:
class RagRetriever:
    """Handles the query based retrievel from the vectordb"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """ Retrieve relevant documents for a query
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # print(f"query_embeddings: Shape -> {query_embedding.shape}")
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # print(f"results: {results}")
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
    

rag_retreiver = RagRetriever(vector_store,embedding_manager)

In [10]:
rag_retreiver
results = rag_retreiver.retrieve("Multi-modal RAG")
results



Retrieving documents for query: 'Multi-modal RAG'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1  texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generate embeddins with shape of (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_58a984b3_190',
  'content': '16\nFig. 6. Summary of RAG ecosystem\ninitial learning curve. 3) Specialization - optimizing RAG to\nbetter serve production environments.\nThe mutual growth of RAG models and their technology\nstacks is evident; technological advancements continuously\nestablish new standards for existing infrastructure. In turn,\nenhancements to the technology stack drive the development\nof RAG capabilities. RAG toolkits are converging into a\nfoundational technology stack, laying the groundwork for\nadvanced enterprise applications. However, a fully integrated,\ncomprehensive platform concept is still in the future, requiring\nfurther innovation and development.\nF . Multi-modal RAG\nRAG has transcended its initial text-based question-\nanswering confines, embracing a diverse array of modal data.\nThis expansion has spawned innovative multimodal models\nthat integrate RAG concepts across various domains:\nImage. RA-CM3 [176] stands as a pioneering multimoda

### Integrating Vector DB context pipeline with LLM Output

In [11]:
from dotenv import load_dotenv
from google import genai


load_dotenv("dotenv_path=.env")  # Load environment variables from .env file
print("Environment variables loaded successfully.")
api_key = os.getenv("GEMINI_API_KEY")

# print(f"GEMINI_API_KEY: {api_key}")  # For debugging purposes,

client  = genai.Client(api_key=api_key)


def getResponse(query:str,top_k:int):
    if len(query) == 0 :
        raise ValueError("Query is required")

    results = rag_retreiver.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}
    Question:
    {query}
    Answer:"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents={"text" : prompt}
    )

    if not response:
        raise ValueError("Something went wrong while generating response")

    return response


res = getResponse("What is RAG?",3)



Environment variables loaded successfully.
Retrieving documents for query: 'What is RAG?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1  texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Generate embeddins with shape of (1, 384)
Retrieved 3 documents (after filtering)


In [12]:
res.candidates[0].content.parts[0].text

'Based on the provided context, RAG (Retrieval-Augmented Generation) is a methodology integrated with Large Language Models (LLMs) that is cost-effective, surpasses the performance of native LLMs, and involves core stages of "Retrieval," "Generation," and "Augmentation." Its research paradigms are continuously evolving and are categorized into three stages: Naive RAG, Advanced RAG, and Modular RAG.'